# MJ AI Assistant — Intent Classifier
### 22-class DistilBERT intent classification
**Dataset:** 22,000 samples | **Model:** distilbert-base-uncased | **Output:** `exports/mj_intent_model/`

In [ ]:
# Cell 1 — Install Required Packages
import subprocess, sys

packages = [
    "transformers>=4.40.0",
    "datasets>=2.18.0",
    "torch>=2.0.0",
    "accelerate>=0.27.0",
    "scikit-learn>=1.3.0",
    "seqeval>=1.2.2",
    "numpy",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
print("All packages installed OK")


In [ ]:
# Cell 2 — Generate Synthetic Dataset  (1 000 samples × 22 intents)
import random, re, json
from collections import Counter
random.seed(42)

YOUTUBE_Q = [
    "trending songs","lofi music","python tutorial","yash toxic trailer",
    "kannada songs","bollywood hits","coding music","gym workout",
    "react tutorial","deep learning","cricket highlights","tamil songs",
    "motivational speech","ted talks","machine learning basics",
    "fastapi tutorial","docker tutorial","web development course",
    "data science","new movies 2024","comedy videos","beats to study",
]
GOOGLE_Q = [
    "VTU results","python documentation","fastapi docs",
    "best laptop under 50000","weather today","stock market today",
    "exam timetable","top engineering colleges","flight tickets",
    "ipl schedule","cricket score","latest news","movie reviews",
    "recipe for biryani","java vs python","best programming languages",
    "coronavirus updates","population of india","how to lose weight",
]
APPS = [
    "VS Code","Notepad","Chrome","Spotify","Discord","Slack",
    "Zoom","VLC","Steam","Task Manager","Paint","Excel",
    "Word","PowerPoint","Telegram","WhatsApp","Calculator",
]
REPOS  = ["mj-assistant","my-portfolio","ai-chatbot","fastapi-backend","react-dashboard","ml-model"]
EMAILS = ["john@gmail.com","boss@company.com","professor@vtu.edu","hr@company.org","client@startup.io"]
TASKS  = [
    "fix the login bug","write unit tests","update README",
    "submit assignment","pay electricity bill","buy groceries",
    "call the doctor","prepare presentation","learn docker",
    "review PR","deploy to production","exercise 30 minutes",
]
FACTS  = [
    "my college is Sri Siddhartha Institute of Technology",
    "my project is MJ AI Assistant",
    "my github username is MJCodes",
    "i am preparing for GATE 2025",
    "my stack is Python FastAPI React",
    "i live in Tumkur Karnataka",
    "my name is Manoj",
]

TEMPLATES = {
"open_browser":        ["open browser","launch browser","start chrome","open chrome","open firefox",
                         "open edge","launch web browser","start internet browser",
                         "open a new browser window","browser please","open chromium"],
"youtube_search":      ["open youtube and search {YQ}","search {YQ} on youtube",
                         "find {YQ} on youtube","play {YQ} on youtube","youtube {YQ}",
                         "youtube search {YQ}","search youtube for {YQ}","look up {YQ} on youtube",
                         "find me {YQ} on youtube","go to youtube and search {YQ}",
                         "i want to watch {YQ}","play {YQ} video","watch {YQ} on youtube",
                         "mj find me {YQ} on youtube","search for {YQ} on youtube please"],
"google_search":       ["google {GQ}","search {GQ} on google","google search {GQ}",
                         "search for {GQ}","look up {GQ}","find information about {GQ}",
                         "can you google {GQ}","search the web for {GQ}","find {GQ} online",
                         "research {GQ}","look for {GQ} on google","google {GQ} for me",
                         "find me results for {GQ}","search {GQ} on the internet"],
"open_application":    ["open {APP}","launch {APP}","start {APP}","run {APP}",
                         "open the {APP} app","can you open {APP}","start {APP} please",
                         "hey mj open {APP}","start the {APP}","bring up {APP}",
                         "open {APP} for me","fire up {APP}","get {APP} running"],
"open_calculator":     ["open calculator","launch calculator","start calculator","open calc",
                         "calculator please","bring up calculator","run calculator",
                         "open windows calculator","hey mj open calculator",
                         "i need calculator","show me calculator","open calculator app"],
"open_notepad":        ["open notepad","launch notepad","start notepad","open text editor",
                         "notepad please","bring up notepad","run notepad",
                         "hey mj open notepad","i need notepad","open notes"],
"open_vscode":         ["open vs code","launch vs code","start vscode","open vscode",
                         "open code editor","hey mj open vs code",
                         "open visual studio code","launch code editor",
                         "start code","run vscode","bring up vs code"],
"open_github":         ["open github","launch github","go to github","open github website",
                         "take me to github","open my github","show me github",
                         "open github in browser","hey mj open github","visit github"],
"github_push":         ["push my code to github","git push","push changes to github",
                         "push to {REPO}","commit and push","push latest changes",
                         "push code to {REPO}","push to main","push to origin",
                         "git push origin main","commit and push to github","push my changes"],
"github_pull":         ["pull from github","git pull","pull latest changes",
                         "pull from {REPO}","git pull origin main",
                         "get latest code from github","sync with github",
                         "pull updates","fetch and merge","download latest code",
                         "pull my code","get latest from {REPO}"],
"github_create_repo":  ["create a github repo called {REPO}","new github repo {REPO}",
                         "make a new repo {REPO}","create repo {REPO}",
                         "initialize github repository {REPO}","setup github repo {REPO}",
                         "create a new repository {REPO}","new repo {REPO}",
                         "make github repo {REPO}","create private repo {REPO}"],
"read_email":          ["check my email","open my inbox","read my emails",
                         "show me my emails","check gmail","open email","read my inbox",
                         "show unread emails","any new emails","check my messages",
                         "what is in my inbox","check email please","view my emails"],
"send_email":          ["send an email to {EM}","email {EM}","compose email to {EM}",
                         "write an email to {EM}","send message to {EM}",
                         "draft an email to {EM}","send {EM} an email",
                         "write to {EM}","contact {EM} by email","mail to {EM}"],
"summarize_email":     ["summarize my emails","summarize my inbox",
                         "give me a summary of my emails","email summary please",
                         "brief me on my emails","what emails are important",
                         "summarize unread messages","give email overview",
                         "quick email summary","email briefing","digest my emails"],
"create_task":         ["create a task to {TASK}","add task {TASK}","new task {TASK}",
                         "add to my todo {TASK}","create todo {TASK}",
                         "remind me to {TASK}","add {TASK} to my tasks",
                         "create task {TASK}","put {TASK} on my list",
                         "schedule task {TASK}","add new task {TASK}","task: {TASK}"],
"update_task":         ["mark task {TASK} as done","complete the task {TASK}",
                         "close task {TASK}","task {TASK} is finished",
                         "mark {TASK} complete","update task {TASK}",
                         "set {TASK} as completed","finish task {TASK}",
                         "task {TASK} done","i finished {TASK} mark done"],
"delete_task":         ["delete task {TASK}","remove task {TASK}","cancel task {TASK}",
                         "delete the {TASK} task","remove {TASK} from list",
                         "i no longer need {TASK}","erase task {TASK}",
                         "drop task {TASK}","clear task {TASK}"],
"workflow_create":     ["create a workflow to {TASK}","automate {TASK}",
                         "set up workflow for {TASK}","build automation for {TASK}",
                         "create automation {TASK}","make a workflow for {TASK}",
                         "automate {TASK} every day","setup daily workflow {TASK}",
                         "create recurring task {TASK}","workflow: {TASK}"],
"workflow_run":        ["run workflow {TASK}","execute workflow {TASK}",
                         "start workflow {TASK}","trigger workflow {TASK}",
                         "run the {TASK} workflow","execute automation {TASK}",
                         "run my daily routine","execute daily workflow",
                         "fire workflow {TASK}","run the automation"],
"remember_fact":       ["{FACT}","remember that {FACT}","note that {FACT}",
                         "remember: {FACT}","save this: {FACT}",
                         "keep in mind {FACT}","store this fact: {FACT}",
                         "hey mj remember {FACT}","i want you to know {FACT}"],
"recall_memory":       ["what is my college","what project am i working on",
                         "where do i study","what is my github",
                         "what am i preparing for","what is my name",
                         "what did i tell you about me","recall my preferences",
                         "what do you know about me","who am i",
                         "what is my stack","where do i work","what do i do"],
"chat":                ["hi","hello","hey mj","how are you","what can you do",
                         "help me","good morning","good evening","good night",
                         "what is your name","tell me a joke","how is it going",
                         "what is up mj","hey there","nice to meet you","who are you",
                         "can you help me","introduce yourself","talk to me",
                         "are you there","wake up mj","what are your features"],
}

TYPOS = {
    "open":["opn","opne","oepn"], "search":["serach","seach","serch"],
    "youtube":["yotube","youtubee","ytube"], "google":["goggle","gogle","googel"],
    "github":["githob","gitub","guthub"], "email":["emial","emaail","emal"],
}

def _pick(lst): return random.choice(lst)

def fill(t):
    t = t.replace("{YQ}",   _pick(YOUTUBE_Q))
    t = t.replace("{GQ}",   _pick(GOOGLE_Q))
    t = t.replace("{APP}",  _pick(APPS))
    t = t.replace("{REPO}", _pick(REPOS))
    t = t.replace("{EM}",   _pick(EMAILS))
    t = t.replace("{TASK}", _pick(TASKS))
    t = t.replace("{FACT}", _pick(FACTS))
    return t

def noise(text):
    if random.random() < 0.10: text = text.upper()
    elif random.random() < 0.20: text = text.title()
    if random.random() < 0.10:
        for w, tl in TYPOS.items():
            if w in text.lower():
                text = re.sub(w, _pick(tl), text, count=1, flags=re.IGNORECASE)
                break
    if random.random() < 0.15:
        text = text + _pick([".", "!", "?", " please"])
    if random.random() < 0.08:
        text = _pick(["mj", "hey mj", "hey"]) + " " + text
    return text.strip()

records = []
SAMPLES = 1000
for intent, tmpl_list in TEMPLATES.items():
    c = 0
    while c < SAMPLES:
        t = noise(fill(_pick(tmpl_list)))
        records.append({"text": t, "intent": intent})
        c += 1

random.shuffle(records)
print(f"Generated {len(records)} samples across {len(TEMPLATES)} intents")
dist = Counter(r["intent"] for r in records)
for k, v in sorted(dist.items()):
    print(f"  {k:<25} {v}")


In [ ]:
# Cell 3 — Save Dataset as JSONL
import json
from pathlib import Path

dataset_dir = Path("datasets")
dataset_dir.mkdir(exist_ok=True)

out = dataset_dir / "mj_intents_22class.jsonl"
with open(out, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Saved {len(records)} records -> {out}  ({out.stat().st_size//1024} KB)")


In [ ]:
# Cell 4 — Load Dataset
import json
from pathlib import Path

loaded = []
with open(Path("datasets/mj_intents_22class.jsonl"), encoding="utf-8") as f:
    for line in f:
        loaded.append(json.loads(line.strip()))

texts  = [r["text"]   for r in loaded]
labels = [r["intent"] for r in loaded]
print(f"Loaded {len(loaded)} samples")
print(f"Sample[0]:    {loaded[0]}")
print(f"Sample[5000]: {loaded[5000]}")


In [ ]:
# Cell 5 — Label Encoding
from sklearn.preprocessing import LabelEncoder
import numpy as np

le = LabelEncoder()
label_ids  = le.fit_transform(labels)
label_names = list(le.classes_)
label2id    = {l: int(i) for i, l in enumerate(label_names)}
id2label    = {int(i): l for l, i in label2id.items()}

print(f"{len(label_names)} intent classes:")
for i, name in enumerate(label_names):
    print(f"  [{i:02d}] {name}")


In [ ]:
# Cell 6 — Train / Validation / Test Split  80/10/10
from sklearn.model_selection import train_test_split

X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    texts, label_ids.tolist(), test_size=0.2, random_state=42, stratify=label_ids
)
X_val, X_te, y_val, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp
)

print(f"Train:      {len(X_tr):>6}")
print(f"Validation: {len(X_val):>6}")
print(f"Test:       {len(X_te):>6}")


In [ ]:
# Cell 7 — Train DistilBERT Intent Classifier
# Expected runtime: ~15 min CPU | ~3 min GPU
import torch, numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN    = 128
BATCH      = 32
EPOCHS     = 5
device     = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device.upper()}")

def mk_ds(X, y): return Dataset.from_dict({"text": X, "label": y})
hf = DatasetDict({
    "train":      mk_ds(X_tr,  y_tr),
    "validation": mk_ds(X_val, y_val),
    "test":       mk_ds(X_te,  y_te),
})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tok(b): return tokenizer(b["text"], truncation=True, max_length=MAX_LEN)
hf_tok = hf.map(tok, batched=True, remove_columns=["text"])

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_names),
    id2label=id2label, label2id=label2id,
)

def metrics(eval_pred):
    logits, lbls = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == lbls).mean())}

from pathlib import Path
chk = Path("models/checkpoints"); chk.mkdir(parents=True, exist_ok=True)

args = TrainingArguments(
    output_dir=str(chk), num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=64,
    learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1,
    evaluation_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="accuracy",
    logging_steps=50, seed=42, fp16=torch.cuda.is_available(), report_to="none",
)
trainer = Trainer(
    model=model, args=args,
    train_dataset=hf_tok["train"], eval_dataset=hf_tok["validation"],
    tokenizer=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=metrics,
)
print("Starting training ...")
res = trainer.train()
print(f"Done! Loss={res.training_loss:.4f}  Time={res.metrics['train_runtime']:.0f}s")


In [ ]:
# Cell 8 — Evaluate on Test Set
from sklearn.metrics import classification_report
import numpy as np

out   = trainer.predict(hf_tok["test"])
preds = np.argmax(out.predictions, axis=-1)
true  = out.label_ids

acc = (preds == true).mean()
print(f"Test Accuracy: {acc:.4f}  ({acc*100:.2f}%)")
print()
print(classification_report(true, preds, target_names=label_names, digits=4))


In [ ]:
# Cell 9 — Save Model
import json
from pathlib import Path

save_dir = Path("exports/mj_intent_model")
save_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(save_dir))
tokenizer.save_pretrained(str(save_dir))

mapping = {
    "label2id":    label2id,
    "id2label":    {str(k): v for k, v in id2label.items()},
    "intent_names": label_names,
    "num_classes": len(label_names),
    "model":       MODEL_NAME,
    "max_length":  MAX_LEN,
}
with open(save_dir / "label_mapping.json", "w") as f:
    json.dump(mapping, f, indent=2)

print(f"Model saved to: {save_dir}")
for p in sorted(save_dir.iterdir()):
    print(f"  {p.name:<40} {p.stat().st_size//1024} KB")


In [ ]:
# Cell 10 — Predictor Function
import torch

def predict_intent(text, top_k=3):
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                        max_length=MAX_LEN, padding=True)
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim=-1)[0]
    top = torch.argsort(probs, descending=True)[:top_k]
    results = [{"intent": id2label[i.item()], "confidence": round(probs[i].item(), 4)} for i in top]
    return {"text": text, "intent": results[0]["intent"],
            "confidence": results[0]["confidence"], "top_k": results}

r = predict_intent("open youtube and search trending songs")
print(f"Test: {r['text']}")
print(f"  -> {r['intent']}  ({r['confidence']:.1%})")


In [ ]:
# Cell 11 — Interactive Testing
TESTS = [
    # ── Desktop / Search ─────────────────────────────────────────────────────
    ("open youtube and search yash toxic trailer",    "youtube_search"),
    ("search trending kannada songs on youtube",      "youtube_search"),
    ("youtube lofi music",                            "youtube_search"),
    ("google VTU results",                            "google_search"),
    ("look up python documentation",                  "google_search"),
    ("open calculator",                               "open_calculator"),
    ("open notepad",                                  "open_notepad"),
    ("open vs code",                                  "open_vscode"),
    ("open github",                                   "open_github"),
    ("open chrome",                                   "open_browser"),
    ("launch spotify",                                "open_application"),
    # ── GitHub ───────────────────────────────────────────────────────────────
    ("push my latest code to github",                "github_push"),
    ("git pull origin main",                          "github_pull"),
    ("create github repo mj-assistant",              "github_create_repo"),
    # ── Email ────────────────────────────────────────────────────────────────
    ("check my email",                               "read_email"),
    ("send email to boss@company.com",               "send_email"),
    ("summarize my inbox",                           "summarize_email"),
    # ── Tasks ────────────────────────────────────────────────────────────────
    ("create a task fix the login bug",              "create_task"),
    ("mark task buy groceries as done",              "update_task"),
    ("delete task pay electricity bill",             "delete_task"),
    # ── Workflow ─────────────────────────────────────────────────────────────
    ("create a workflow to send daily report",       "workflow_create"),
    ("run my daily routine",                         "workflow_run"),
    # ── Memory ───────────────────────────────────────────────────────────────
    ("my college is Sri Siddhartha Institute",       "remember_fact"),
    ("what is my college",                           "recall_memory"),
    # ── Chat ─────────────────────────────────────────────────────────────────
    ("hi",                                           "chat"),
    ("what can you do",                              "chat"),
    # ── Typos & noise ────────────────────────────────────────────────────────
    ("opn youtube serach trending songs",            "youtube_search"),
    ("serach for VTU results on goggle",             "google_search"),
    ("hey mj open calclator",                        "open_calculator"),
]

print(f"{'Input':<50} {'Expected':<25} {'Predicted':<25} Conf  OK?")
print("─" * 115)
ok = 0
for text, exp in TESTS:
    r = predict_intent(text)
    hit = r["intent"] == exp
    if hit: ok += 1
    mark = "OK" if hit else "FAIL"
    print(f"{text[:48]:<50} {exp:<25} {r['intent']:<25} {r['confidence']:>4.0%}  {mark}")

print(f"\nScore: {ok}/{len(TESTS)} = {ok/len(TESTS):.1%}")


In [ ]:
# Cell 12 — Export to MJ Backend
import shutil
from pathlib import Path

src = Path("exports/mj_intent_model")
dst = Path("../backend/app/ml_models/mj_intent_model")
dst.mkdir(parents=True, exist_ok=True)

for f in src.iterdir():
    shutil.copy2(f, dst / f.name)

print(f"Model exported to MJ backend: {dst.resolve()}")
print("Files:")
for f in sorted(dst.iterdir()):
    print(f"  {f.name}")


In [ ]:
# Cell 13 — FastAPI ControllerAgent Integration

CODE = """
# app/ml/intent_predictor.py

import json, time, torch
from functools import lru_cache
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = Path(__file__).parent.parent / "ml_models" / "mj_intent_model"

@lru_cache(maxsize=1)
def _load():
    tok   = AutoTokenizer.from_pretrained(str(MODEL_PATH))
    mdl   = AutoModelForSequenceClassification.from_pretrained(str(MODEL_PATH))
    mdl.eval()
    with open(MODEL_PATH / "label_mapping.json") as f:
        m = json.load(f)
    id2label = {int(k): v for k, v in m["id2label"].items()}
    return tok, mdl, id2label

def predict_intent(text: str, threshold: float = 0.5) -> dict:
    t0 = time.monotonic()
    tok, mdl, id2label = _load()
    inp = tok(text, return_tensors="pt", truncation=True, max_length=128, padding=True)
    with torch.no_grad():
        probs = torch.softmax(mdl(**inp).logits, dim=-1)[0]
    top3  = torch.argsort(probs, descending=True)[:3]
    top_k = [{"intent": id2label[i.item()], "confidence": round(probs[i].item(), 4)} for i in top3]
    best  = top_k[0]
    return {
        "intent":     best["intent"] if best["confidence"] >= threshold else "chat",
        "confidence": best["confidence"],
        "top_k":      top_k,
        "latency_ms": round((time.monotonic() - t0) * 1000, 2),
    }
"""
print("=== app/ml/intent_predictor.py ===")
print(CODE)

PATCH = """
# In ControllerAgent.execute() — drop-in fast routing:

from app.ml.intent_predictor import predict_intent as ml_predict

async def execute(self, state):
    ml = ml_predict(state.raw_input)
    if ml["confidence"] >= 0.85:          # high-confidence fast path
        return {
            "intent": ml["intent"],
            "context": {}, "plan": [],
            "risk_level": "safe",
            "requires_approval": False,
            "agent_logs": [f"[controller:ml] {ml['intent']} {ml['confidence']:.0%} {ml['latency_ms']}ms"],
        }
    return await self._llm_intent_detection(state)   # fallback to LLM
"""
print("=== ControllerAgent patch ===")
print(PATCH)


In [ ]:
# Cell 14 — Performance Benchmark
import time, statistics, torch

model.eval()
INPUTS = [
    "open youtube search trending songs",
    "google VTU results",
    "open calculator",
    "push my code to github",
    "check my email",
    "create task fix login bug",
    "hi",
    "my college is Sri Siddhartha",
]
latencies = []
for i in range(200):
    t = INPUTS[i % len(INPUTS)]
    s = time.perf_counter()
    predict_intent(t)
    latencies.append((time.perf_counter() - s) * 1000)

latencies.sort()
print(f"Latency over 200 requests ({device.upper()}):")
print(f"  Min:    {latencies[0]:.2f} ms")
print(f"  Median: {statistics.median(latencies):.2f} ms")
print(f"  P95:    {latencies[189]:.2f} ms")
print(f"  Max:    {latencies[-1]:.2f} ms")
print(f"  Mean:   {statistics.mean(latencies):.2f} ms")
print()
med = statistics.median(latencies)
print(f"Throughput:   ~{int(1000/med)} req/s")
print(f"vs LLM API:   ~{int(2500/med)}x faster")
print()
print("=" * 50)
print("  Model saved to: exports/mj_intent_model/")
print("  Ready for MJ AI Assistant production!")
print("=" * 50)
